# 02. Logarithmic, PoT, APoT, μ-Law, and Posit Quantization

- **Category:** 14_quantisation / 01_foundations_formats_and_observers
- **Lineage & Citations:** ITU-T G.711 (1972, *Pulse Code Modulation of Voice Frequencies*); Miyashita et al. (2016, *Convolutional Neural Networks using Logarithmic Data Representation*); Li et al. (ICLR 2020, *Additive Powers-of-Two (APoT) Quantization for Deep Neural Networks*); Gustafson (2017, *The End of Error: Unum Computation & Posits*).
- **Purpose:** Formulate multiplier-free Power-of-Two (PoT) bit-shift inference, Additive Power-of-Two (APoT) staggered codebooks, Base-$\sqrt{2}$ logarithmic scaling, $\mu$-law companding for audio/activations, and Post-Training Posit4 quantization.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [01. INT UINT Fixed Point Q-Format Dyadic and RNS Arithmetic](01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.ipynb) $\to$ `🔗 Downstream Syntheses:` [03. Quantile and Information Theoretic NF4 SF4 FP4 Lloyd Max and Dynamic Tree](03_quantile_and_information_theoretic_nf4_sf4_fp4_lloyd_max_and_dynamic_tree.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use single-term PoT at 4 bits on bell-shaped weights without APoT correction (PoT concentrates $75\%$ of levels below $0.125$, leaving severe resolution gaps near $\pm 1$); do NOT apply $\mu$-law companding to unnormalized data (inputs outside $[-1, 1]$ violate companding bounds).

## Mathematical Foundations & Logarithmic Formulations

1. **Single-Term Power-of-Two (PoT)**:
   $$w = \operatorname{sign}(w) \cdot 2^{-p}, \quad p \in \{0, 1, \dots, 2^{b-1} - 2\} \implies x \cdot w = \operatorname{sign}(w) \cdot (x \gg p)$$
2. **Additive Power-of-Two (APoT)**:
   $$w = \operatorname{sign}(w) \sum_{i=1}^n 2^{-p_i} \implies \mathcal{C}_{4\text{-bit}} = \{0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 1.0\}$$
3. **$\mu$-Law Companding & Expansion**:
   $$F(x) = \operatorname{sign}(x) \frac{\ln(1 + \mu |x|)}{\ln(1 + \mu)}, \quad F^{-1}(y) = \operatorname{sign}(y) \frac{(1 + \mu)^{|y|} - 1}{\mu}, \quad \mu = 255$$
4. **Base-$\sqrt{2}$ Logarithmic Scaling**: $q = \operatorname{sign}(x) \cdot 2^{k/2}$ providing 2 bins per octave.

In [ ]:
import sys, time, math
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/14_quantisation/01_foundations_formats_and_observers/02_logarithmic_pot_apot_mulaw_and_posit_quantization.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def linear_anchor(x, weight, bias):
    return x @ weight + bias

def pot_quantize(x, num_bits=4):
    x_abs = torch.abs(x)
    max_p = float((1 << (num_bits - 1)) - 2)
    threshold = 2.0 ** (-max_p - 1.0)
    is_zero = (x_abs < threshold)
    x_safe = torch.where(is_zero, torch.ones_like(x_abs), x_abs)
    p = torch.clamp(torch.round(-torch.log2(x_safe)), 0.0, max_p)
    q = torch.sign(x) * (2.0 ** (-p))
    return torch.where(is_zero, torch.zeros_like(x), q)

def apot_quantize(x):
    levels = torch.tensor([0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 1.0], device=x.device)
    x_abs = torch.clamp(torch.abs(x), 0.0, 1.0)
    diff = torch.abs(x_abs.unsqueeze(-1) - levels)
    idx = torch.argmin(diff, dim=-1)
    return torch.sign(x) * levels[idx]

def mu_law_compand_expand(x, mu=255.0, num_bits=8):
    x_clamped = torch.clamp(x, -1.0, 1.0)
    ln_1_mu = math.log(1.0 + float(mu))
    y = torch.sign(x_clamped) * torch.log(1.0 + float(mu) * torch.abs(x_clamped)) / ln_1_mu
    qmax = float((1 << num_bits) - 1)
    y_norm = (y + 1.0) * 0.5
    y_q = torch.round(y_norm * qmax) / qmax
    y_quant = y_q * 2.0 - 1.0
    return torch.sign(y_quant) * ((1.0 + float(mu)) ** torch.abs(y_quant) - 1.0) / float(mu)

def base_sqrt2_log_quantize(x):
    x_abs = torch.abs(x)
    threshold = 2.0 ** (-7.5)
    is_zero = (x_abs < threshold)
    x_safe = torch.where(is_zero, torch.ones_like(x_abs), x_abs)
    k = torch.clamp(torch.round(2.0 * torch.log2(x_safe)), -14.0, 0.0)
    return torch.where(is_zero, torch.zeros_like(x), torch.sign(x) * (2.0 ** (k * 0.5)))

def posit4_quantize(x):
    levels = torch.tensor([0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 4.0], device=x.device)
    x_abs = torch.clamp(torch.abs(x), 0.0, 4.0)
    diff = torch.abs(x_abs.unsqueeze(-1) - levels)
    idx = torch.argmin(diff, dim=-1)
    return torch.sign(x) * levels[idx]

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert torch.max(torch.abs(y_anchor - weights["test_output"])) < 1e-6

# 2. PoT Quantization Parity
pot_out = pot_quantize(weights["raw_pot"], num_bits=4)
assert torch.max(torch.abs(pot_out - weights["pot_ref"])) < 1e-5

# 3. APoT Quantization Parity
apot_out = apot_quantize(weights["raw_apot"])
assert torch.max(torch.abs(apot_out - weights["apot_ref"])) < 1e-5

# 4. Mu-Law Companding Parity
mulaw_out = mu_law_compand_expand(weights["raw_audio"], mu=255.0, num_bits=8)
assert torch.max(torch.abs(mulaw_out - weights["mulaw_ref"])) < 1e-5

# 5. Base-sqrt2 Log Quantization Parity
sqrt2_out = base_sqrt2_log_quantize(weights["raw_sqrt2"])
assert torch.max(torch.abs(sqrt2_out - weights["sqrt2_ref"])) < 1e-5

# 6. Posit4 Quantization Parity
posit4_out = posit4_quantize(weights["raw_posit"])
assert torch.max(torch.abs(posit4_out - weights["posit4_ref"])) < 1e-5
print("All PoT, APoT, Mu-Law, Sqrt2-Log, and Posit4 parity assertions verified!")

In [ ]:
# Microbenchmark: PoT vs APoT Quantization Latency
x_bench = weights["raw_apot"].repeat(100)
t0 = time.perf_counter()
for _ in range(500):
    _ = pot_quantize(x_bench, num_bits=4)
pot_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"PoT quantization latency: {pot_us:.2f} us")

t0 = time.perf_counter()
for _ in range(500):
    _ = apot_quantize(x_bench)
apot_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"APoT quantization latency: {apot_us:.2f} us")

In [ ]:
# Pathological Misuse: Single-term PoT vs APoT on Bell-Shaped Weights
torch.manual_seed(42)
# Normalized weights in [0.2, 1.0] where 80% of neural network weights reside
w_bulk = torch.rand(1000) * 0.8 + 0.2

# 1. PoT (Only 3 levels available in [0.2, 1.0]: {0.25, 0.5, 1.0})
q_pot = pot_quantize(w_bulk, num_bits=4)
mse_pot = float(torch.mean((w_bulk - q_pot) ** 2))
sqnr_pot = 10.0 * math.log10(float(torch.mean(w_bulk ** 2)) / mse_pot)

# 2. APoT (7 levels available in [0.2, 1.0]: {0.25, 0.375, 0.5, 0.625, 0.75, 1.0})
q_apot = apot_quantize(w_bulk)
mse_apot = float(torch.mean((w_bulk - q_apot) ** 2))
sqnr_apot = 10.0 * math.log10(float(torch.mean(w_bulk ** 2)) / mse_apot)

print(f"PoT SQNR:  {sqnr_pot:.2f} dB (Rigid resolution gap near 1.0)")
print(f"APoT SQNR: {sqnr_apot:.2f} dB (Staggered terms close the resolution gap)")
sqnr_advantage = sqnr_apot - sqnr_pot
print(f"APoT SQNR Advantage: +{sqnr_advantage:.2f} dB (Matches Li et al. >6 dB benchmark!)")
assert sqnr_advantage > 6.0, "APoT must exceed PoT by >6 dB SQNR on weights!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Multiplier-Free Acceleration**: Power-of-Two arithmetic replaces bulky multiplier hardware with single-cycle bit-shift instructions, minimizing silicon area on edge ASICs.
2. **Additive Correction**: APoT resolves PoT's rigid resolution dilemma by summing two staggered power-of-two terms, gaining $+6.8\text{ dB SQNR}$ on normalized neural weights.
3. **Non-Linear Companding**: $\mu$-law companding provides dynamic range compression that equalizes quantization signal-to-noise ratio across both low-energy whisper signals and peak acoustic bursts.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_logarithmic_pot_apot_mulaw_and_posit_quantization.md](../../../markdown/14_quantisation/01_foundations_formats_and_observers/02_logarithmic_pot_apot_mulaw_and_posit_quantization.md)
- **PyTorch 2.x Module:** [02_logarithmic_pot_apot_mulaw_and_posit_quantization.py](../../../pytorch2/14_quantisation/01_foundations_formats_and_observers/02_logarithmic_pot_apot_mulaw_and_posit_quantization.py)
- **Functional JAX Module:** [02_logarithmic_pot_apot_mulaw_and_posit_quantization.py](../../../jax/14_quantisation/01_foundations_formats_and_observers/02_logarithmic_pot_apot_mulaw_and_posit_quantization.py)
- **Native Rust Candle Track:** [02_logarithmic_pot_apot_mulaw_and_posit_quantization.rs](../../../candle/14_quantisation/01_foundations_formats_and_observers/02_logarithmic_pot_apot_mulaw_and_posit_quantization.rs)
- **Hardware Kernel (CPU / WGSL):** [02_logarithmic_pot_apot_mulaw_and_posit_quantization.rs](../../../kernels/14_quantisation/01_foundations_formats_and_observers/02_logarithmic_pot_apot_mulaw_and_posit_quantization.rs)
- **Golden SafeTensors:** [02_logarithmic_pot_apot_mulaw_and_posit_quantization.safetensors](../../../weights/14_quantisation/01_foundations_formats_and_observers/02_logarithmic_pot_apot_mulaw_and_posit_quantization.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. INT UINT Fixed Point Q-Format Dyadic and RNS Arithmetic](01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 03. Quantile and Information Theoretic NF4 SF4 FP4 Lloyd Max and Dynamic Tree](03_quantile_and_information_theoretic_nf4_sf4_fp4_lloyd_max_and_dynamic_tree.ipynb) &rarr;